# 🔍 SQL vs Elasticsearch — Mismos datos, distintas preguntas

**Workshop: Más allá de SQL** | ITESM — Inteligencia de Negocios

En este notebook vamos a conectarnos a un clúster de Elasticsearch con **10,000+ órdenes de eCommerce** (el dataset de Kibana) y cargar los mismos datos en **SQLite**. Después, ejecutaremos las mismas preguntas de negocio en ambos motores para ver dónde cada uno brilla — y dónde se queda corto.

### Qué vas a aprender

| Ronda | Pregunta de negocio | SQL | ES |
|-------|-------------------|-----|----|
| 1 | Búsqueda de productos por texto | `LIKE` | `match` |
| 2 | Tolerancia a typos | ❌ No puede | `fuzziness` |
| 3 | Ranking por relevancia | Binario (sí/no) | BM25 score |
| 4 | Búsqueda en múltiples campos | Múltiples `OR` | `multi_match` |
| 5 | Agregaciones con contexto textual | `GROUP BY` | `aggs` + text |
| 6 | Filtro + texto + ranking combinado | Complejo e ineficiente | `bool` query |
| 7 | Preguntas en lenguaje natural | ❌ Imposible | ES AI Assistant |

---

**Requisitos**: Una API key de Elastic Cloud (se genera en el workshop).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/mas-alla-de-sql/blob/main/labs/lab-es-vs-sql.ipynb)

In [ ]:
# Instalar dependencias
!pip install -q elasticsearch

## Sección 0 — Conexión a Elasticsearch

Vamos a usar la **API key** que generaste en Elastic Cloud.

**Cómo obtener tu API key:**
1. Entra a tu deployment en Elastic Cloud
2. Ve a **Management → Stack Management → API keys**
3. Click en **Create API key**
4. Nombre: `workshop-itesm` (o lo que quieras)
5. Copia el valor encoded (el string largo)

También necesitas el **Elasticsearch endpoint** (URL) de tu deployment.

In [ ]:
from getpass import getpass

# --- Configuración de conexión ---
# Pega tu endpoint de Elastic Cloud (ej: https://mi-deployment.es.us-central1.gcp.cloud.es.io)
ES_ENDPOINT = input("🔗 Elasticsearch endpoint (URL): ").strip()

# Pega tu API key encoded
ES_API_KEY = getpass("🔑 API key (encoded): ")

In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    ES_ENDPOINT,
    api_key=ES_API_KEY,
)

# Verificar conexión
info = es.info()
print(f"✅ Conectado a Elasticsearch {info['version']['number']}")
print(f"   Cluster: {info['cluster_name']}")

In [ ]:
# Verificar que el índice eCommerce existe
INDEX = "kibana_sample_data_ecommerce"

if es.indices.exists(index=INDEX):
    count = es.count(index=INDEX)["count"]
    print(f"✅ Índice '{INDEX}' encontrado — {count:,} documentos")
else:
    print(f"❌ Índice '{INDEX}' no encontrado.")
    print("   Ve a Kibana Home → 'Try sample data' → 'Sample eCommerce orders' → Add data")
    print("   Después vuelve a ejecutar esta celda.")

## Sección 1 — Descargar datos de ES a SQLite

Vamos a extraer **todos los documentos** del índice eCommerce de ES y cargarlos en una base de datos SQLite local. Así podemos correr las mismas preguntas en ambos motores.

Esto simula lo que harías en una empresa: el mismo catálogo de órdenes vive en una BD relacional (para reportes) y en Elasticsearch (para búsqueda).

In [ ]:
import json

def fetch_all_docs(es_client, index, batch_size=1000):
    """Descarga todos los documentos de un indice ES usando scroll API."""
    docs = []
    resp = es_client.search(
        index=index,
        body={"query": {"match_all": {}}, "size": batch_size},
        scroll="2m",
    )
    scroll_id = resp["_scroll_id"]
    hits = resp["hits"]["hits"]
    docs.extend(hits)

    while len(hits) > 0:
        resp = es_client.scroll(scroll_id=scroll_id, scroll="2m")
        scroll_id = resp["_scroll_id"]
        hits = resp["hits"]["hits"]
        docs.extend(hits)

    es_client.clear_scroll(scroll_id=scroll_id)
    return docs

print(f"Descargando documentos de '{INDEX}'...")
raw_docs = fetch_all_docs(es, INDEX)
print(f"✅ {len(raw_docs):,} documentos descargados")

In [ ]:
# Ver la estructura de un documento
sample = raw_docs[0]["_source"]
print("Campos disponibles:")
for key in sorted(sample.keys()):
    val = sample[key]
    if isinstance(val, list) and len(val) > 0 and isinstance(val[0], dict):
        print(f"  {key}: [{', '.join(val[0].keys())}] (lista de objetos)")
    else:
        short_val = str(val)[:60]
        print(f"  {key}: {short_val}")

In [ ]:
import sqlite3
import pandas as pd

# --- Aplanar documentos para SQLite ---
# El dataset de eCommerce tiene productos como nested objects.
# Para SQLite, aplanamos: una fila por orden con los datos del primer producto.

rows = []
for doc in raw_docs:
    src = doc["_source"]
    # Tomar el primer producto (las ordenes pueden tener 1-4 productos)
    product = src.get("products", [{}])[0] if src.get("products") else {}
    # Concatenar nombres de todos los productos para busqueda textual
    all_product_names = "; ".join(
        p.get("product_name", "") for p in src.get("products", [])
    )

    rows.append({
        "order_id": doc["_id"],
        "customer_full_name": src.get("customer_full_name", ""),
        "customer_gender": src.get("customer_gender", ""),
        "email": src.get("email", ""),
        "day_of_week": src.get("day_of_week", ""),
        "order_date": src.get("order_date", ""),
        "currency": src.get("currency", ""),
        "taxful_total_price": src.get("taxful_total_price", 0),
        "taxless_total_price": src.get("taxless_total_price", 0),
        "total_quantity": src.get("total_quantity", 0),
        "total_unique_products": src.get("total_unique_products", 0),
        # Producto principal
        "product_name": product.get("product_name", ""),
        "category": product.get("category", ""),
        "base_price": product.get("base_price", 0),
        "discount_percentage": product.get("discount_percentage", 0),
        "quantity": product.get("quantity", 0),
        "manufacturer": product.get("manufacturer", ""),
        # Todos los productos (para busqueda textual)
        "all_product_names": all_product_names,
        # Geo
        "continent": src.get("geoip", {}).get("continent_name", ""),
        "country": src.get("geoip", {}).get("country_iso_code", ""),
        "city": src.get("geoip", {}).get("city_name", ""),
    })

df = pd.DataFrame(rows)
print(f"✅ DataFrame: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head(3)

In [ ]:
# --- Cargar en SQLite ---
conn = sqlite3.connect(":memory:")
df.to_sql("orders", conn, index=False, if_exists="replace")

# Verificar
result = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
print(f"✅ SQLite: tabla 'orders' con {result:,} filas")

# Muestra de columnas
cols = conn.execute("PRAGMA table_info(orders)").fetchall()
print(f"\nColumnas ({len(cols)}):")
for col in cols:
    print(f"  {col[1]} ({col[2]})")

### ✅ Listo: mismos datos en dos motores

Ahora tenemos:
- **Elasticsearch**: índice `kibana_sample_data_ecommerce` con todos los campos originales (nested objects, texto analizado, índice invertido)
- **SQLite**: tabla `orders` con los mismos datos aplanados (filas y columnas, texto sin analizar)

Vamos a hacerle las mismas preguntas a cada uno.

---

## Ronda 1 — Búsqueda de texto: `LIKE` vs `match`

**Pregunta de negocio**: *¿Qué órdenes incluyen zapatos?*

Un analista de eCommerce hace esta pregunta todos los días.

In [ ]:
# --- SQL: LIKE ---
print("=" * 70)
print("🗄️  SQL: SELECT ... WHERE product_name LIKE '%shoe%'")
print("=" * 70)

sql_query = """
SELECT order_id, customer_full_name, product_name, taxful_total_price
FROM orders
WHERE all_product_names LIKE '%shoe%'
LIMIT 5
"""
sql_results = pd.read_sql(sql_query, conn)
print(f"Resultados: {len(sql_results)}")
print(sql_results.to_string(index=False))

# Contar total
total_sql = conn.execute(
    "SELECT COUNT(*) FROM orders WHERE all_product_names LIKE '%shoe%'"
).fetchone()[0]
print(f"\nTotal coincidencias SQL: {total_sql}")

In [ ]:
# --- Elasticsearch: match ---
print("=" * 70)
print("🔍 ES: match { 'products.product_name': 'shoes' }")
print("=" * 70)

es_resp = es.search(
    index=INDEX,
    body={
        "query": {
            "match": {
                "products.product_name": "shoes"
            }
        },
        "size": 5,
        "_source": ["customer_full_name", "products.product_name", "taxful_total_price"],
    },
)

print(f"Total coincidencias ES: {es_resp['hits']['total']['value']:,}")
print(f"\nTop 5 (ordenados por relevancia):")
for hit in es_resp["hits"]["hits"]:
    src = hit["_source"]
    score = hit["_score"]
    products = ", ".join(p["product_name"] for p in src.get("products", []))
    print(f"  ⭐ {score:.2f} | {src.get('customer_full_name', '')} | {products} | ${src.get('taxful_total_price', 0):.2f}")

### 💡 Observa

- SQL usa `LIKE '%shoe%'` — búsqueda de **substring** (caso exacto, sin ranking)
- ES usa `match` — **tokeniza** "shoes" → ["shoe"] (stemming), busca en el índice invertido, y **rankea** por BM25
- ES devuelve un **score de relevancia** (⭐). SQL solo devuelve sí/no
- ES encuentra "shoes", "Shoe", "SHOES" automáticamente. SQL necesitaría `LOWER()` o `COLLATE NOCASE`

---

## Ronda 2 — Tolerancia a typos: SQL no puede, ES sí

**Pregunta de negocio**: Un usuario escribe "shoees" en el buscador de la tienda. *¿Qué debería pasar?*

En eCommerce, el 10-15% de las búsquedas tienen errores de dedo. Si tu buscador no tolera typos, pierdes ventas.

In [ ]:
# --- SQL: no puede manejar typos ---
print("=" * 70)
print("🗄️  SQL: LIKE '%shoees%' (con typo)")
print("=" * 70)

typo_sql = conn.execute(
    "SELECT COUNT(*) FROM orders WHERE all_product_names LIKE '%shoees%'"
).fetchone()[0]
print(f"Resultados: {typo_sql}")
print("❌ SQL no encuentra nada. El usuario ve una página vacía.")

In [ ]:
# --- ES: fuzzy search tolera typos ---
print("=" * 70)
print("🔍 ES: match con fuzziness: 'shoees'")
print("=" * 70)

es_fuzzy = es.search(
    index=INDEX,
    body={
        "query": {
            "match": {
                "products.product_name": {
                    "query": "shoees",
                    "fuzziness": "AUTO",
                }
            }
        },
        "size": 5,
        "_source": ["customer_full_name", "products.product_name", "taxful_total_price"],
    },
)

print(f"Total coincidencias ES: {es_fuzzy['hits']['total']['value']:,}")
for hit in es_fuzzy["hits"]["hits"]:
    src = hit["_source"]
    products = ", ".join(p["product_name"] for p in src.get("products", []))
    print(f"  ⭐ {hit['_score']:.2f} | {products}")

print("\n✅ ES corrigió el typo 'shoees' → 'shoes' automáticamente.")
print("   Esto es distancia de Levenshtein bajo el capot.")

In [ ]:
# Más typos que ES maneja y SQL no
typos = ["botas", "jaket", "T-shrt", "pantalon"]

print("\n🎯 Tolerancia a typos — ES vs SQL:")
print(f"{'Typo':<12} {'SQL (LIKE)':<15} {'ES (fuzzy)':<15}")
print("-" * 42)

for typo in typos:
    sql_count = conn.execute(
        f"SELECT COUNT(*) FROM orders WHERE all_product_names LIKE '%{typo}%'"
    ).fetchone()[0]

    es_count = es.count(
        index=INDEX,
        body={
            "query": {
                "match": {
                    "products.product_name": {
                        "query": typo,
                        "fuzziness": "AUTO",
                    }
                }
            }
        },
    )["count"]

    print(f"{typo:<12} {sql_count:<15} {es_count:<15}")

### 💡 ¿Por qué importa?

- En eCommerce, un buscador que no tolera typos **pierde ventas directamente**
- Amazon, Mercado Libre, Liverpool — todos usan fuzzy search
- SQL no tiene esta capacidad de forma nativa. Tendrías que implementar Levenshtein a mano o usar extensiones externas

---

## Ronda 3 — Ranking por relevancia

**Pregunta de negocio**: *Búscame órdenes de ropa de mujer. ¿Cuáles son las más relevantes?*

SQL devuelve resultados en orden arbitrario (o por alguna columna). ES los **rankea por relevancia** usando BM25.

In [ ]:
# --- SQL: sin ranking, orden arbitrario ---
print("=" * 70)
print("🗄️  SQL: LIKE '%Women%Clothing%' — sin ranking")
print("=" * 70)

sql_women = pd.read_sql("""
SELECT order_id, product_name, category, taxful_total_price
FROM orders
WHERE category LIKE '%Women%Clothing%'
LIMIT 5
""", conn)

print("Resultados (orden arbitrario, sin score):")
print(sql_women.to_string(index=False))
print("\n⚠️  SQL no sabe cuál resultado es 'mejor'. Solo filtra sí/no.")

In [ ]:
# --- ES: ranking BM25 ---
print("=" * 70)
print("🔍 ES: match 'Women Clothing' — con ranking BM25")
print("=" * 70)

es_women = es.search(
    index=INDEX,
    body={
        "query": {
            "multi_match": {
                "query": "Women Clothing",
                "fields": ["products.category", "products.product_name"],
            }
        },
        "size": 5,
        "_source": ["customer_full_name", "products.product_name", "products.category", "taxful_total_price"],
    },
)

print("Resultados (ordenados por relevancia BM25):")
for hit in es_women["hits"]["hits"]:
    src = hit["_source"]
    products = ", ".join(p.get("product_name", "") for p in src.get("products", []))
    categories = ", ".join(p.get("category", "") for p in src.get("products", []))
    print(f"  ⭐ {hit['_score']:.2f} | {products[:50]} | {categories}")

print("\n✅ ES rankea: los resultados donde 'Women' y 'Clothing' aparecen juntos")
print("   tienen score más alto que donde aparecen separados.")

### 💡 BM25 en 30 segundos

BM25 es el algoritmo de ranking que usa ES (y Google). Asigna un score basado en:

1. **TF** (Term Frequency): ¿Cuántas veces aparece el término en el documento?
2. **IDF** (Inverse Document Frequency): ¿Qué tan raro es el término en todos los documentos?
3. **Longitud del documento**: Documentos más cortos con el término son más relevantes

SQL no tiene concepto de "relevancia". Solo sabe: ¿está o no está?

---

## Ronda 4 — Búsqueda en múltiples campos

**Pregunta de negocio**: *Busca "leather" en cualquier campo: nombre del producto, categoría, fabricante...*

En SQL necesitas escribir un `OR` por cada campo. En ES es una línea.

In [ ]:
# --- SQL: múltiples OR ---
print("=" * 70)
print("🗄️  SQL: múltiples LIKE con OR")
print("=" * 70)

sql_multi = pd.read_sql("""
SELECT order_id, product_name, category, manufacturer
FROM orders
WHERE product_name LIKE '%leather%'
   OR category LIKE '%leather%'
   OR manufacturer LIKE '%leather%'
   OR all_product_names LIKE '%leather%'
LIMIT 5
""", conn)

total_multi_sql = conn.execute("""
SELECT COUNT(*) FROM orders
WHERE product_name LIKE '%leather%'
   OR category LIKE '%leather%'
   OR manufacturer LIKE '%leather%'
   OR all_product_names LIKE '%leather%'
""").fetchone()[0]

print(f"Total: {total_multi_sql}")
print(sql_multi.to_string(index=False))
print("\n⚠️  4 líneas de WHERE. Si agregas un campo nuevo, hay que modificar la query.")

In [ ]:
# --- ES: multi_match ---
print("=" * 70)
print("🔍 ES: multi_match en 3 campos (con boosting)")
print("=" * 70)

es_multi = es.search(
    index=INDEX,
    body={
        "query": {
            "multi_match": {
                "query": "leather",
                "fields": [
                    "products.product_name^2",  # nombre vale doble
                    "products.category",
                    "products.manufacturer",
                ],
            }
        },
        "size": 5,
        "_source": ["products.product_name", "products.category", "products.manufacturer", "taxful_total_price"],
    },
)

print(f"Total: {es_multi['hits']['total']['value']:,}")
for hit in es_multi["hits"]["hits"]:
    src = hit["_source"]
    products = ", ".join(p.get("product_name", "") for p in src.get("products", []))
    print(f"  ⭐ {hit['_score']:.2f} | {products[:60]}")

print("\n✅ Una sola query busca en todos los campos.")
print("   product_name^2 = si aparece en el nombre, vale el doble de score.")

### 💡 Field boosting

El `^2` en `products.product_name^2` significa: *si "leather" aparece en el nombre del producto, dale el doble de importancia que si aparece en la categoría*.

Esto no existe en SQL. En SQL todos los campos tienen el mismo peso.

---

## Ronda 5 — Agregaciones: `GROUP BY` vs `aggs`

**Pregunta de negocio**: *¿Cuál es el revenue promedio por categoría de producto?*

Aquí SQL brilla — es exactamente para lo que fue diseñado. Pero ES también puede hacerlo, y además combinarlo con búsqueda de texto.

In [ ]:
# --- SQL: GROUP BY (territorio natural de SQL) ---
print("=" * 70)
print("🗄️  SQL: GROUP BY category — aquí SQL brilla")
print("=" * 70)

sql_agg = pd.read_sql("""
SELECT
    category,
    COUNT(*) as ordenes,
    ROUND(AVG(taxful_total_price), 2) as precio_promedio,
    ROUND(SUM(taxful_total_price), 2) as revenue_total
FROM orders
WHERE category != ''
GROUP BY category
ORDER BY revenue_total DESC
LIMIT 10
""", conn)

print(sql_agg.to_string(index=False))
print("\n✅ SQL es claro, legible y rápido para agregaciones simples.")

In [ ]:
# --- ES: aggs (equivalente, más verbose pero más flexible) ---
print("=" * 70)
print("🔍 ES: aggs por categoría")
print("=" * 70)

es_agg = es.search(
    index=INDEX,
    body={
        "size": 0,
        "aggs": {
            "por_categoria": {
                "terms": {
                    "field": "products.category.keyword",
                    "size": 10,
                    "order": {"revenue_total": "desc"},
                },
                "aggs": {
                    "precio_promedio": {"avg": {"field": "taxful_total_price"}},
                    "revenue_total": {"sum": {"field": "taxful_total_price"}},
                },
            }
        },
    },
)

print(f"{'Categoría':<30} {'Ordenes':<10} {'Precio Prom':<15} {'Revenue Total':<15}")
print("-" * 70)
for bucket in es_agg["aggregations"]["por_categoria"]["buckets"]:
    print(
        f"{bucket['key']:<30} "
        f"{bucket['doc_count']:<10} "
        f"${bucket['precio_promedio']['value']:<14.2f} "
        f"${bucket['revenue_total']['value']:<14.2f}"
    )

print("\n🤝 Mismo resultado. SQL es más conciso para esto.")
print("   Pero ES puede combinar aggs + búsqueda textual (próxima ronda).")

### 💡 Scocard honesto

| Capacidad | SQL | ES |
|-----------|-----|----|
| `GROUP BY` simple | ✅ Más limpio | ✅ Funciona, más verbose |
| JOINs | ✅ Nativo | ❌ No tiene joins |
| Aggs + texto ("revenue de productos que contienen 'boot'") | ⚠️ Posible con `LIKE`, sin ranking | ✅ Nativo y rankeado |
| Aggs en tiempo real sobre millones de docs | ⚠️ Depende del índice | ✅ Diseñado para esto |

---

## Ronda 6 — La query killer: filtro + texto + ranking

**Pregunta de negocio**: *Encuentra órdenes de más de $75 donde el producto sea tipo "boot" o "leather", en órdenes de viernes, y rankea por relevancia.*

Esta es la query donde ES realmente brilla. Combina:
- Filtro numérico (precio > $75)
- Filtro exacto (día = viernes)
- Búsqueda de texto con relevancia ("boot" OR "leather")
- Ranking BM25

In [ ]:
# --- SQL: se puede, pero sin ranking ---
print("=" * 70)
print("🗄️  SQL: filtro + texto combinado (sin ranking)")
print("=" * 70)

sql_combo = pd.read_sql("""
SELECT order_id, customer_full_name, product_name,
       taxful_total_price, day_of_week
FROM orders
WHERE taxful_total_price > 75
  AND day_of_week = 'Friday'
  AND (all_product_names LIKE '%boot%' OR all_product_names LIKE '%leather%')
ORDER BY taxful_total_price DESC
LIMIT 5
""", conn)

print(sql_combo.to_string(index=False))
print(f"\n⚠️  SQL ordena por precio, no por relevancia textual.")
print("   No sabe si 'boot' es más relevante que 'leather' en cada resultado.")

In [ ]:
# --- ES: bool query (filtro + texto + ranking) ---
print("=" * 70)
print("🔍 ES: bool query — filtro + texto + ranking")
print("=" * 70)

es_combo = es.search(
    index=INDEX,
    body={
        "query": {
            "bool": {
                "must": [
                    {
                        "multi_match": {
                            "query": "boot leather",
                            "fields": ["products.product_name^2", "products.category"],
                        }
                    }
                ],
                "filter": [
                    {"range": {"taxful_total_price": {"gt": 75}}},
                    {"term": {"day_of_week": "Friday"}},
                ],
            }
        },
        "size": 5,
        "_source": [
            "customer_full_name", "products.product_name",
            "taxful_total_price", "day_of_week",
        ],
    },
)

print(f"Total: {es_combo['hits']['total']['value']}")
for hit in es_combo["hits"]["hits"]:
    src = hit["_source"]
    products = ", ".join(p.get("product_name", "") for p in src.get("products", []))
    print(
        f"  ⭐ {hit['_score']:.2f} | "
        f"${src.get('taxful_total_price', 0):.2f} | "
        f"{src.get('day_of_week', '')} | "
        f"{products[:55]}"
    )

print("\n✅ ES hace todo en una query:")
print("   1. Filtra precio > $75 y día = Friday (como SQL WHERE)")
print("   2. Busca 'boot' y 'leather' en texto (como full-text search)")
print("   3. Rankea por relevancia BM25 (imposible en SQL)")

### 💡 La clave: `must` vs `filter`

```
bool:
  must:    → Afecta el SCORE (ranking). Usa esto para texto.
  filter:  → NO afecta el score. Usa esto para filtros exactos (precio, fecha).
  should:  → "Bonus" al score. Resultados que cumplen ganan puntos extra.
  must_not:→ Excluye. Como WHERE NOT en SQL.
```

---

## Ronda 7 — Highlighting: ES muestra POR QUÉ encontró el resultado

**Pregunta de negocio**: *Busca "bag" en productos y muéstrame dónde está la coincidencia.*

Como el resaltado de Google cuando buscas algo. SQL no tiene esto.

In [ ]:
# --- ES: highlight ---
print("=" * 70)
print("🔍 ES: highlight — resalta las coincidencias")
print("=" * 70)

es_hl = es.search(
    index=INDEX,
    body={
        "query": {
            "match": {
                "products.product_name": "bag"
            }
        },
        "highlight": {
            "fields": {
                "products.product_name": {}
            }
        },
        "size": 5,
        "_source": ["products.product_name", "taxful_total_price"],
    },
)

for hit in es_hl["hits"]["hits"]:
    highlights = hit.get("highlight", {}).get("products.product_name", [])
    hl_text = " | ".join(highlights) if highlights else "(sin highlight)"
    print(f"  ⭐ {hit['_score']:.2f} | {hl_text}")

print("\n✅ Las palabras entre <em>...</em> son las que coincidieron.")
print("   En una UI las verías resaltadas en amarillo (como Google).")
print("   SQL no tiene equivalente nativo.")

---

## Resumen: ¿Cuándo usar cada uno?


In [ ]:
# Tabla resumen visual
print("""
┌─────────────────────────────────────────┬─────────┬─────────┐
│ Capacidad                               │   SQL   │   ES    │
├─────────────────────────────────────────┼─────────┼─────────┤
│ Filtros exactos (=, >, <, BETWEEN)       │  ✅✅✅  │  ✅✅   │
│ JOINs entre tablas                       │  ✅✅✅  │  ❌     │
│ GROUP BY / agregaciones simples           │  ✅✅✅  │  ✅✅   │
│ Transacciones ACID                       │  ✅✅✅  │  ❌     │
│ Búsqueda de texto (relevancia)           │  ❌     │  ✅✅✅  │
│ Tolerancia a typos (fuzzy)               │  ❌     │  ✅✅✅  │
│ Ranking por relevancia (BM25)            │  ❌     │  ✅✅✅  │
│ Highlighting de coincidencias            │  ❌     │  ✅✅✅  │
│ Multi-field search + boosting            │  ⚠️     │  ✅✅✅  │
│ Filtro + texto + ranking combinado       │  ⚠️     │  ✅✅✅  │
│ Escala a millones de docs en tiempo real │  ⚠️     │  ✅✅✅  │
│ Preguntas en lenguaje natural            │  ❌     │  ✅✅✅  │
└─────────────────────────────────────────┴─────────┴─────────┘

💡 No es SQL vs ES. Es SQL + ES.
   Las empresas modernas usan AMBOS: SQL para reportes y transacciones,
   ES para búsqueda, observabilidad y análisis de texto.
""")

---

## Bonus — ES AI Assistant: preguntas en lenguaje natural

Elasticsearch tiene un **AI Assistant** integrado en Kibana que usa un LLM para responder preguntas en lenguaje natural sobre tus datos. No necesitas escribir queries — solo preguntas como si hablaras con un analista.

### ¿Cómo acceder?

En Kibana, busca el icono de **AI Assistant** (bot) en la barra superior derecha, o ve a **Observability → AI Assistant**.

### Preguntas para hacerle al AI Assistant

Estas son preguntas que puedes copiar y pegar directamente en el AI Assistant de Kibana para demostrar su poder con el dataset de eCommerce:

#### Pregunta 1 — Análisis de ventas

```
Using the kibana_sample_data_ecommerce index, what are the top 5 product
categories by total revenue this week? Show me the trend compared to
last week.
```

**¿Por qué es impactante?** El LLM traduce tu pregunta a una query de ES con agregaciones, la ejecuta, y te devuelve la respuesta en lenguaje natural. Un analista de BI tardaría 15-20 minutos escribiendo el query a mano.

#### Pregunta 2 — Detección de anomalías

```
Look at the kibana_sample_data_ecommerce index. Are there any unusual
patterns in order values? For example, orders with unusually high
totals or unusual product combinations.
```

**¿Por qué es impactante?** Pedir "patrones inusuales" en SQL requiere que tú definas qué es "inusual" (desviaciones estándar, percentiles, etc.). El LLM decide la mejor estrategia analítica por ti.

#### Pregunta 3 — Segmentación de clientes

```
In kibana_sample_data_ecommerce, which customers have made the most
orders? What product categories do they prefer? Is there a difference
in spending patterns between male and female customers?
```

**¿Por qué es impactante?** Son 3 preguntas encadenadas que requerirían 3+ queries SQL con JOINs y subqueries. El AI Assistant las resuelve en una sola conversación.

#### Pregunta 4 — Generación de código ES

```
Write me an Elasticsearch query for kibana_sample_data_ecommerce that
finds all orders containing "shoes" with a total price over $100,
grouped by manufacturer, showing average price per manufacturer.
```

**¿Por qué es impactante?** Ahora el AI Assistant es tu **tutor de ES**. En vez de buscar en la documentación, le describes lo que quieres y te genera la query lista para Dev Tools.

#### Pregunta 5 — Exploración abierta

```
I'm new to this ecommerce dataset. Can you give me a summary of
what's in kibana_sample_data_ecommerce? How many orders, what date
range, what kinds of products, and what are the most interesting
patterns you can find?
```

**¿Por qué es impactante?** Le estás pidiendo que haga **análisis exploratorio** (EDA) de tus datos. Esto sería 30+ minutos de queries SQL y gráficas en Python. El AI Assistant lo resume en segundos.

### 💡 ¿Por qué el AI Assistant puede hacer esto y SQL no?

El AI Assistant de ES funciona con **RAG** (la misma técnica que vemos en el workshop):

1. Tu pregunta en lenguaje natural llega al LLM
2. El LLM conoce el schema de tus índices (los campos, tipos de datos)
3. Genera una o más queries de ES
4. Ejecuta las queries contra tu clúster
5. Interpreta los resultados y te responde en lenguaje natural

SQL podría tener algo similar (ej: text2sql), pero ES tiene ventaja porque:
- Ya indexa texto de forma inteligente (no necesita `LIKE`)
- Las agregaciones combinan con búsqueda textual nativamente
- El schema flexible (JSON) hace más fácil la exploración

---

## Conclusiones

| Lo que viste | Lo que significa para BI |
|-------------|------------------------|
| `match` vs `LIKE` | ES entiende texto, SQL solo busca substrings |
| `fuzziness` | ES tolera errores humanos, SQL no |
| BM25 score | ES rankea por relevancia, SQL devuelve sí/no |
| `multi_match` + boosting | ES busca en múltiples campos con pesos, SQL necesita múltiples `OR` |
| `bool` query | ES combina filtros + texto + ranking en una query |
| `highlight` | ES muestra por qué encontró cada resultado |
| AI Assistant | ES responde preguntas en lenguaje natural |

### El mensaje final

**SQL no va a desaparecer**. Sigue siendo la mejor herramienta para reportes, transacciones y datos relacionales.

Pero si tu empresa tiene:
- Un buscador de productos
- Logs de servidores
- Texto libre (quejas, chats, reseñas)
- Necesidad de búsqueda rápida sobre millones de documentos

...entonces necesitas algo **más allá de SQL**.

📁 `github.com/HesusG/mas-alla-de-sql`